In [19]:
from datetime import date, datetime, timedelta

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_ai_model.validation import flights_df_comparison
from data_engineering.adsbx.readsb import read_adsbx_readsb
from flights.flights_match_adsb import get_matching_icaos_in_flights
import polars as pl
DAY_MARGIN = timedelta(minutes=60)

test_date = date(2026,3,1)
df_adsb = read_adsbx_readsb()
df_gold = get_bts_flights_for_day(test_date)
df_gold = get_matching_icaos_in_flights(df_gold, df_adsb)
# df_test = get_adsbx_flights_for_day(test_date)
df_test = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')

expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
min_time = datetime(test_date.year, test_date.month, test_date.day) + DAY_MARGIN
max_time = datetime(test_date.year, test_date.month, test_date.day) + timedelta(days=1) - DAY_MARGIN
expr2 = (pl.col("takeoff_time") > min_time) & (pl.col("landing_time") < max_time)
expr = expr & expr2
df_gold = df_gold.filter(expr)
df_test = df_test.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
df,stats = flights_df_comparison(df_test, df_gold)

6 overlapping flights found for 2026-03-01

--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 1787
Predicted flights: 5710  |  Gold flights: 5719
Correctly matched: 5700  |  Predicted unmatched: 10  |  Gold flights missed: 19
Precision: 99.82%  |  Recall: 99.67%  |  F1: 99.75%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 5688/5689 matched (99.98%)
landing_airport_ident: 5677/5677 matched (100.00%)
overall: 11365/11366 matched (99.99%)


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_56214/858160093.py:25: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))


In [23]:
COLUMNS = ["icao", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df2", "takeoff_airport_ident_df2", "landing_time_df2","landing_airport_ident_df2", "match_status"]
df = df.select(COLUMNS)

In [27]:
df.filter(pl.col("match_status") == "df2_only")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a1e6b9""",null,null,null,null,2026-03-01 01:03:00,"""KELP""",2026-03-01 02:32:00,"""KHOU""","""df2_only"""
"""a2b8b8""",null,null,null,null,2026-03-01 01:36:00,"""PHNL""",2026-03-01 09:30:00,"""PGUM""","""df2_only"""
"""a3ac77""",null,null,null,null,2026-03-01 01:01:00,"""KSLC""",2026-03-01 03:10:00,"""KSAT""","""df2_only"""
"""a5add3""",null,null,null,null,2026-03-01 01:06:00,"""KDEN""",2026-03-01 01:48:00,"""KCPR""","""df2_only"""
"""a6b435""",null,null,null,null,2026-03-01 01:23:00,"""PANC""",2026-03-01 02:40:00,"""PAOM""","""df2_only"""
…,…,…,…,…,…,…,…,…,…
"""abe448""",null,null,null,null,2026-03-01 01:02:00,"""KDEN""",2026-03-01 02:28:00,"""KCID""","""df2_only"""
"""ac033e""",null,null,null,null,2026-03-01 01:05:00,"""KDEN""",2026-03-01 02:28:00,"""KDFW""","""df2_only"""
"""ac2813""",null,null,null,null,2026-03-01 01:03:00,"""KSLC""",2026-03-01 02:29:00,"""KSMF""","""df2_only"""


In [28]:
df.filter(pl.col("icao") == "a1e6b9").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a1e6b9""",null,null,null,null,2026-03-01 01:03:00,"""KELP""",2026-03-01 02:32:00,"""KHOU""","""df2_only"""
"""a1e6b9""",2026-03-01 03:46:12.314,"""KHOU""",2026-03-01 04:20:43.771,"""KCRP""",2026-03-01 03:46:00,"""KHOU""",2026-03-01 04:20:00,"""KCRP""","""matched"""
"""a1e6b9""",2026-03-01 11:49:28.642,"""KCRP""",2026-03-01 12:23:43.988,"""KHOU""",2026-03-01 11:49:00,"""KCRP""",2026-03-01 12:23:00,"""KHOU""","""matched"""
"""a1e6b9""",2026-03-01 13:37:11.616,"""KHOU""",2026-03-01 14:21:27.603,"""KDAL""",2026-03-01 13:37:00,"""KHOU""",2026-03-01 14:19:00,"""KDAL""","""matched"""
"""a1e6b9""",2026-03-01 15:21:28.277,"""KDAL""",2026-03-01 16:41:12.926,"""KSTL""",2026-03-01 15:21:00,"""KDAL""",2026-03-01 16:41:00,"""KSTL""","""matched"""
"""a1e6b9""",2026-03-01 18:04:55.807,"""KSTL""",2026-03-01 20:00:13.737,"""KLGA""",2026-03-01 18:05:00,"""KSTL""",2026-03-01 20:00:00,"""KLGA""","""matched"""


In [26]:
df.filter(pl.col("icao") == "a0b59d").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a0b59d""",2026-03-01 16:08:36.994,"""KSLC""",2026-03-01 17:48:11.812,"""KPDX""",2026-03-01 16:13:00,"""KSLC""",2026-03-01 17:48:00,"""KPDX""","""matched"""
"""a0b59d""",2026-03-01 18:47:43.245,"""KPDX""",2026-03-01 20:10:58.403,"""KSLC""",2026-03-01 18:47:00,"""KPDX""",2026-03-01 20:10:00,"""KSLC""","""matched"""


In [8]:
df_gold.filter(pl.col("icao") == "a19982")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a19982""",null,"""N202AS""",2026-03-01 01:51:00,"""KPDX""",2026-03-01 07:46:00,"""PHNL""",false,false,false,false,null,null,null,"""A3"""
